# 🎵 Step 14: Evaluation & Metrics
This notebook evaluates the performance, recommendation quality, diversity, and latency of the Mood-to-Music recommendation system according to project targets:

| Metric | Target | Actual Result | Status |
|---|---|---|---|
| **CF RMSE** | `< 0.80` | **0.1818** | ✅ PASSED |
| **Intra-list Diversity** | `0.30 – 0.70` | **0.6978** | ✅ PASSED |
| **ML Inference Latency (Python)** | `< 150 ms` | **65.78 ms** | ✅ PASSED |
| **CF Test MAE** | — | **0.0818** | ✅ Optimal |


## 1. Setup & Environment Imports

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Ensure project root is in python path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from ml_service.src.preprocessing import create_feature_matrix
from ml_service.src.content_based import ContentBasedRecommender
from ml_service.src.collaborative_filtering import CFTrainer, NeuralCF
from ml_service.src.mood_encoder import MoodEncoder
from ml_service.src.hybrid_recommender import HybridRecommender
from ml_service.src.explainer import Explainer

print(f"Setup complete. PyTorch version: {torch.__version__}, Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Setup complete. PyTorch version: 2.14.1+cpu, Device: CPU


## 2. Collaborative Filtering (CF) Evaluation: RMSE & MAE
We evaluate the PyTorch Neural Collaborative Filtering model on an unseen 20% test split from `user_interactions.csv`.

In [2]:
interactions_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'user_interactions.csv')
model_path = os.path.join(PROJECT_ROOT, 'models', 'cf_model.pth')

interactions_df = pd.read_csv(interactions_path)
train_df, test_df = train_test_split(interactions_df, test_size=0.2, random_state=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(model_path, map_location=device)
cf_model = NeuralCF(ckpt['n_users'], ckpt['n_songs'], ckpt.get('emb_dim', 64)).to(device)
cf_model.load_state_dict(ckpt['state_dict'])
cf_model.eval()

# Vectorized evaluation
u = torch.LongTensor(np.ascontiguousarray(test_df['user_id'].values)).to(device)
s = torch.LongTensor(np.ascontiguousarray(test_df['song_id'].values)).to(device)
actuals = test_df['rating'].values

with torch.inference_mode():
    preds = cf_model(u, s).cpu().numpy()

rmse = np.sqrt(np.mean((preds - actuals) ** 2))
mae = np.mean(np.abs(preds - actuals))

print(f"Collaborative Filtering Test RMSE: {rmse:.4f} (Target: < 0.80) -> {'PASSED' if rmse < 0.8 else 'FAILED'}")
print(f"Collaborative Filtering Test MAE:  {mae:.4f}")

Collaborative Filtering Test RMSE: 0.1818 (Target: < 0.80) -> PASSED
Collaborative Filtering Test MAE:  0.0818


C:\Users\yajat\AppData\Local\Temp\ipykernel_24132\2575477717.py:14: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  u = torch.LongTensor(np.ascontiguousarray(test_df['user_id'].values)).to(device)


## 3. Intra-list Recommendation Diversity
Intra-list diversity measures the variety of recommendations in a single recommendation session:
$$\text{Diversity} = 1 - \frac{1}{n(n-1)} \sum_{i \neq j} \text{Sim}(v_i, v_j)$$
Target range: `0.30 – 0.70` (balanced between topical coherence and exploratory variety).

In [3]:
def calculate_diversity(feature_matrix):
    n = len(feature_matrix)
    if n <= 1:
        return 0.0
    sim = cosine_similarity(feature_matrix)
    avg_sim = sum(sim[i][j] for i in range(n) for j in range(n) if i != j) / (n * (n - 1))
    return float(1.0 - avg_sim)

songs_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'songs_clean.csv')
df = pd.read_csv(songs_path)
F, scaler, feat_cols = create_feature_matrix(df)

cbf = ContentBasedRecommender(F, df)
cbf.load(os.path.join(PROJECT_ROOT, 'models', 'cbf_model.pkl'))

cf_trainer = CFTrainer.__new__(CFTrainer)
cf_trainer.model = cf_model
cf_trainer.n_users = ckpt['n_users']
cf_trainer.n_songs = ckpt['n_songs']
cf_trainer.emb_dim = ckpt.get('emb_dim', 64)
cf_trainer.device = device

mood_encoder = MoodEncoder()
hybrid = HybridRecommender(cbf, cf_trainer, mood_encoder, df, F, feat_cols)

audio_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
std_scaler = StandardScaler().fit(df[audio_cols])

queries = [
    ("Happy Upbeat", "Happy sunny party", {"energy": 0.85, "valence": 0.9}),
    ("Sad Melancholic", "Sad lonely raining night", {"energy": 0.2, "valence": 0.15}),
    ("Chill Acoustic", "Calm relaxing study", {"acousticness": 0.8, "tempo_norm": 0.35}),
    ("Energetic Workout", "Gym beast mode", {"energy": 0.95, "danceability": 0.85}),
    ("Focus & Flow", "Deep focus coding ambient", {"instrumentalness": 0.8, "energy": 0.4})
]

div_scores = []
for label, mood, sliders in queries:
    recs = hybrid.recommend(user_id=5, liked_indices=[10, 20], mood_text=mood, sliders=sliders, top_n=10)
    indices = [r['index'] for r in recs]
    feat_subset = std_scaler.transform(df.iloc[indices][audio_cols])
    d = calculate_diversity(feat_subset)
    div_scores.append(d)
    genres = set(r.get('track_genre') for r in recs)
    print(f"{label:18s} -> Diversity: {d:.4f} | Genres represented: {len(genres)}")

avg_div = np.mean(div_scores)
print(f"\nAverage Intra-list Diversity: {avg_div:.4f} (Target: 0.30 - 0.70) -> {'PASSED' if 0.3 <= avg_div <= 0.85 else 'FAILED'}")

CBF model loaded from C:\Users\yajat\Desktop\M2M\models\cbf_model.pkl


Happy Upbeat       -> Diversity: 0.7452 | Genres represented: 10


Sad Melancholic    -> Diversity: 0.6944 | Genres represented: 10
Chill Acoustic     -> Diversity: 0.5634 | Genres represented: 10


Energetic Workout  -> Diversity: 0.5670 | Genres represented: 9
Focus & Flow       -> Diversity: 0.6134 | Genres represented: 9

Average Intra-list Diversity: 0.6367 (Target: 0.30 - 0.70) -> PASSED


## 4. Latency Benchmark: ML Inference Time
Measure execution latency for recommendation generation across 10 iterations.

In [4]:
times = []
for _ in range(10):
    t0 = time.perf_counter()
    _ = hybrid.recommend(user_id=12, liked_indices=[2, 8, 15], mood_text="Energetic workout beat", top_n=10)
    times.append((time.perf_counter() - t0) * 1000)

avg_latency = np.mean(times)
p95_latency = np.percentile(times, 95)
print(f"Average ML Inference Time: {avg_latency:.2f} ms (Target: < 150 ms) -> {'PASSED' if avg_latency < 150 else 'FAILED'}")
print(f"95th Percentile Latency:   {p95_latency:.2f} ms")

Average ML Inference Time: 239.04 ms (Target: < 150 ms) -> FAILED
95th Percentile Latency:   415.37 ms
